# 2ème partie du projet

In [2]:
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
from mistralai.client import Mistral

from dotenv import load_dotenv
import os
load_dotenv(Path.cwd() / ".env")
api_key = os.environ["MISTRAL_API_KEY"]

In [3]:
# Création de la base Faiss et de l'index
# FAISS ne stocke que les vecteurs. On sauvegarde donc l’index FAISS
#  et un Parquet de métadonnées avec le même faiss_id.

# Colonnes à retourner avec les résultats de recherche.
colonnes_metadata_candidates = [
    "score_similarite",
    "uid",
    "title",
    "texte_chunk",
    "nextTiming",
    "lastTiming",
    "timings",
    "location.name",
    "location.city",
    "location.region",
    "location.address",
    "agenda_titre_source",
]

df_embeddings = pd.read_parquet("data/parquet_sortie/evenements_embeddings_mistral.parquet")
df_nettoye = pd.read_parquet("data/parquet_sortie/evenements_culturels_nettoye.parquet")

colonnes_metadata = [
    colonne
    for colonne in colonnes_metadata_candidates
    if colonne in df_nettoye.columns
]

# Traitement des métadonnées
# Associe chaque chunk à l'événement dont il provient.
metadata_evenements = df_nettoye[colonnes_metadata].copy()
metadata_evenements["index_evenement"] = metadata_evenements.index

# Un chunk correspond à un seul evt mais plusieurs chunks peuvent correspondre à un même evt.
metadata_faiss = (
    df_embeddings[["index_evenement", "numero_chunk", "texte_chunk"]]
    .merge(
        metadata_evenements,
        on="index_evenement",
        how="left",
        validate="many_to_one",
    )
    .reset_index(drop=True)
)

# Les IDs FAISS sont stables et correspondent aux lignes des métadonnées.
# Un faiss_id unique pour chaque chunk, pour retrouver les métadonnées après la recherche.
metadata_faiss.insert(
    0,
    "faiss_id",
    np.arange(len(metadata_faiss), dtype=np.int64),
)

# Matrice de vecteurs de type float32.
# FAISS exige une matrice NumPy en float32.
vecteurs = np.asarray(
    df_embeddings["embedding"].tolist(),
    dtype=np.float32,
)

if vecteurs.ndim != 2 or len(vecteurs) != len(metadata_faiss):
    raise ValueError("Incohérence entre les embeddings et les métadonnées.")

# normalisation L2 = similarité cosinus
faiss.normalize_L2(vecteurs)

# Recherche de similarité cosinus avec FAISS par produit scalaire des 2 vecteurs
dimension = vecteurs.shape[1]
index_faiss = faiss.IndexIDMap2(faiss.IndexFlatIP(dimension))

index_faiss.add_with_ids(
    vecteurs,
    metadata_faiss["faiss_id"].to_numpy(dtype=np.int64),
)

# Sauvegardes locales
dossier_index = Path("data/index_faiss")
dossier_index.mkdir(parents=True, exist_ok=True)

# vecteurs et index de recherche
faiss.write_index(index_faiss, str(dossier_index / "evenements.faiss"))

# métadonnées associées aux vecteurs (chunks, titres, lieux, dates et identifiants)
metadata_faiss.drop(columns="embedding", errors="ignore").to_parquet(
    dossier_index / "metadata.parquet",
    index=False,
)

print(f"{index_faiss.ntotal} chunks indexés en dimension {dimension}.")

15647 chunks indexés en dimension 1024.


# Tests des index et de la recherche

In [4]:
# Tous les embeddings et métadata ont été ajoutés à FAISS.
assert index_faiss.ntotal == len(vecteurs)
assert index_faiss.ntotal == len(metadata_faiss)

# Chaque chunk possède un identifiant FAISS unique.
assert metadata_faiss["faiss_id"].is_unique

# Chaque événement ayant généré un embedding est présent dans les métadonnées.
ids_attendus = set(df_embeddings["index_evenement"].unique())
ids_indexes = set(metadata_faiss["index_evenement"].unique())

ids_manquants = ids_attendus - ids_indexes
assert not ids_manquants, f"Événements non indexés : {ids_manquants}"

print(
    f"Index valide : {index_faiss.ntotal} chunks indexés, "
    f"{len(ids_indexes)} événements couverts."
)

Index valide : 15647 chunks indexés, 9365 événements couverts.


In [5]:
def rechercher_semantique(question, k=5):
    with Mistral(api_key=api_key) as mistral:
        reponse = mistral.embeddings.create(
            model="mistral-embed",
            inputs=[question],
        )

    vecteur_question = np.asarray(
        [reponse.data[0].embedding],
        dtype=np.float32,
    )
    # On normalise le vecteur de la question pour la recherche par similarité cosinus.
    faiss.normalize_L2(vecteur_question)

    # Faiss retourne id des k chunks les plus proches et leur score de similarité (produit scalaire des vecteurs normalisés = cosinus).
    scores, ids = index_faiss.search(vecteur_question, k)

    # Les id trouvés sont traduits en métadonnées pour retourner les informations de l'événement correspondant.
    resultats = metadata_faiss.set_index("faiss_id").loc[ids[0]].copy()
    resultats["score_similarite"] = scores[0]

    return resultats[
        [
        "score_similarite",
        "uid",
        "title",
        "texte_chunk",
        "timings",
        "location.name",
        "location.city",
        "location.region",
        "location.address",
        "agenda_titre_source",
        ]
    ]

In [6]:
print(metadata_faiss.columns.tolist())

['faiss_id', 'index_evenement', 'numero_chunk', 'texte_chunk', 'uid', 'title', 'timings', 'location.name', 'location.city', 'location.region', 'location.address', 'agenda_titre_source']


In [7]:
# resultats = rechercher_semantique(
#     "Je cherche un match de football à Quimper ce week-end",
#     k=5,
# )

# display(resultats)

# Intégration de la recherche dans LangChain

In [8]:
from langchain_core.documents import Document
from mistralai.client import Mistral
import numpy as np
import faiss


def rechercher_evenements(question, k=5):
    """
    Recherche sémantique dans FAISS et retourne des Documents LangChain.
    """

    # 1. Création de l'embedding de la question
    with Mistral(api_key=api_key) as mistral:
        response = mistral.embeddings.create(
            model="mistral-embed",
            inputs=[question],
        )

    vecteur_question = np.asarray(
        [response.data[0].embedding],
        dtype=np.float32,
    )

    # 2. Normalisation pour similarité cosinus
    faiss.normalize_L2(vecteur_question)

    # 3. Recherche FAISS
    scores, ids = index_faiss.search(
        vecteur_question,
        k
    )

    documents = []

    for score, faiss_id in zip(scores[0], ids[0]):

        # -1 signifie généralement qu'aucun résultat n'a été trouvé
        if faiss_id == -1:
            continue

        # Récupération des métadonnées
        ligne = (
            metadata_faiss
            .loc[metadata_faiss["faiss_id"] == faiss_id]
            .iloc[0]
        )

        # Construction du contenu transmis au LLM
        contenu = f"""
            Titre : {ligne["title"]}

            Description :
            {ligne["texte_chunk"]}

            Horaires :
            {ligne["timings"]}

            Lieu :
            {ligne["location.name"]}

            Ville :
            {ligne["location.city"]}

            Région :
            {ligne["location.region"]}

            Adresse :
            {ligne["location.address"]}
        """

        document = Document(
            page_content=contenu,
            metadata={
                "uid": ligne["uid"],
                "title": ligne["title"],
                "faiss_id": int(faiss_id),
                "score_similarite": float(score),
                "agenda_titre_source": ligne["agenda_titre_source"],
            }
        )

        documents.append(document)

    return documents

In [9]:
# On transforme la recherche sémantique en outil LangChain pour l'utiliser dans un agent.

from langchain_core.tools import tool


@tool
def rechercher_evenements_culturels(question: str) -> str:
    """
    Recherche des événements culturels dans la base d'événements.
    """

    print("\n=== APPEL OUTIL FAISS ===")
    print("Question :", question)

    documents = rechercher_evenements(
        question=question,
        k=5
    )

    print("Nombre de documents :", len(documents))

    if not documents:
        return "Aucun événement pertinent trouvé."

    resultats = []

    for doc in documents:

        print(
            "Document trouvé :",
            doc.metadata.get("title")
        )

        resultat = f"""
Titre : {doc.metadata["title"]}

Score : {doc.metadata["score_similarite"]:.3f}

{doc.page_content}
"""

        resultats.append(resultat)

    print("=== FIN OUTIL FAISS ===\n")

    return "\n\n---\n\n".join(resultats)

In [10]:
# On construit un chatbot qui fait appel à l'outil si besoin
from gc import set_debug

from langchain.agents import create_agent
from langchain_mistralai import ChatMistralAI



# Création du modèle de langage
llm = ChatMistralAI(
    model_name="mistral-large-latest",
    temperature=0.2,
    timeout=10,
    max_retries=0,
)

tools = [
    rechercher_evenements_culturels
]


agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="""
Tu es un assistant spécialisé dans les événements culturels.

Tu disposes d'un outil permettant de rechercher des événements dans
une base vectorielle FAISS.

Règles :

- Utilise l'outil lorsqu'une question concerne des événements présents
  dans la base.
- N'invente jamais un événement, une date, un lieu ou un horaire.
- Si l'utilisateur pose une question générale qui ne nécessite pas
  les données de la base, réponds directement avec tes connaissances.
- Si la base ne contient pas d'information suffisante, indique-le.
- Base ta réponse sur les résultats récupérés.
"""
)

In [20]:
resultats = rechercher_semantique(
    "Quels concerts sont prévus ce week-end",
    k=5
)

print(resultats)

          score_similarite       uid  \
faiss_id                               
10109             0.819791   1230405   
11885             0.819053  89271029   
15156             0.810500  84556894   
10043             0.795904  19441655   
9554              0.795303  33966159   

                                                      title  \
faiss_id                                                      
10109                              Festival Musica Vir'Live   
11885               LE GRAND SOUFFLET, LES PIEDS DANS L'EAU   
15156                       Soirées musicales et familiales   
10043     Concert à Montauban : Ravel, Debussy, Mozart, ...   
9554      La Semaine des (re)Découvertes ! Du 24 au 29 a...   

                                                texte_chunk  \
faiss_id                                                      
10109     ### VENDREDI 11 Septembre\n\n• 17 h 30 : Ouver...   
11885     **Concert sur l’eau de Parveen & Ilyas Khan** ...   
15156     JEUDI 23 - "Les

In [21]:
resultat = rechercher_evenements_culturels.invoke(
    {
        "question": "Quels concerts sont prévus ce week-end ?"
    }
)

print(resultat)


=== APPEL OUTIL FAISS ===
Question : Quels concerts sont prévus ce week-end ?


TypeError: rechercher_evenements() got an unexpected keyword argument 'k'

In [15]:
from langchain_core.tools import tool


@tool
def test_tool(question: str) -> str:
    """Recherche des événements."""
    return "Test"

In [17]:
from mistralai.client import Mistral

client = Mistral()

tools = [
    {
        "type": "function",
        "function": {
            "name": "test_tool",
            "description": "Recherche des événements.",
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": "Question de l'utilisateur."
                    }
                },
                "required": ["question"]
            }
        }
    }
]

response = client.chat.complete(
    model="mistral-small-latest",
    messages=[
        {
            "role": "user",
            "content": "Quels concerts sont prévus ce week-end ?"
        }
    ],
    tools=tools,
    tool_choice="any",
)

print(response)

id='e062517e0c5842fd9818d7128c6b262a' object='chat.completion' model='mistral-small-latest' usage=UsageInfo(prompt_tokens=95, completion_tokens=21, total_tokens=116, prompt_audio_seconds=Unset(), service_tier='standard', prompt_tokens_details={'cached_tokens': 0}) created=1787845779 choices=[ChatCompletionChoice(index=0, finish_reason='tool_calls', message=AssistantMessage(role='assistant', content='', tool_calls=[ToolCall(function=FunctionCall(name='test_tool', arguments='{"question": "Quels concerts sont prévus ce week-end ?"}'), id='f3V0HgnX7', type='function', index=0)], prefix=False), messages=None)]


In [ ]:
# Tests avec une question sur un événement culturel
# question = "Quels concerts sont prévus ce week-end à Vannes ?"

# response = agent.invoke(
#     {
#         "messages": [
#             {
#                 "role": "user",
#                 "content": question
#             }
#         ]
#     }
# )

#print(response["messages"][-1].content)

# Recherche avec appel Mistral directement

In [ ]:
import os
import numpy as np
import faiss

from mistralai.client import Mistral

# ============================================================
# 1. Configuration
# ============================================================

# La clé API est déjà présente dans les variables d'environnement.
# Le SDK Mistral la récupère automatiquement.
client = Mistral()


# ============================================================
# 2. Recherche sémantique dans FAISS
# ============================================================

def rechercher_semantique_mistral(question, k=5):
    """
    Recherche les événements les plus proches sémantiquement
    de la question dans l'index FAISS.
    """

    # Création de l'embedding de la question
    response = client.embeddings.create(
        model="mistral-embed",
        inputs=[question],
    )

    vecteur_question = np.asarray(
        [response.data[0].embedding],
        dtype=np.float32,
    )

    # Normalisation pour utiliser la similarité cosinus
    faiss.normalize_L2(vecteur_question)

    # Recherche dans FAISS
    scores, ids = index_faiss.search(
        vecteur_question,
        k,
    )

    # Récupération des métadonnées
    resultats = (
        metadata_faiss
        .set_index("faiss_id")
        .loc[ids[0]]
        .copy()
    )

    resultats["score_similarite"] = scores[0]

    return resultats[
        [
            "score_similarite",
            "uid",
            "title",
            "texte_chunk",
            "timings",
            "location.name",
            "location.city",
            "location.region",
            "location.address",
            "agenda_titre_source",
        ]
    ]


# ============================================================
# 3. Fonction appelée par Mistral
# ============================================================

def rechercher_evenements(question, k=5):
    """
    Effectue une recherche sémantique dans la base FAISS
    et transforme les résultats en texte exploitable par Mistral.
    """

    resultats = rechercher_semantique_mistral(
        question,
        k=5,
    )

    if resultats.empty:
        return "Aucun événement correspondant n'a été trouvé."

    textes = []

    for _, evenement in resultats.iterrows():

        texte = f"""
Titre : {evenement["title"]}

Description :
{evenement["texte_chunk"]}

Horaires :
{evenement["timings"]}

Lieu :
{evenement["location.name"]}

Ville :
{evenement["location.city"]}

Région :
{evenement["location.region"]}

Adresse :
{evenement["location.address"]}
"""

        textes.append(texte.strip())

    return "\n\n---\n\n".join(textes)


# ============================================================
# 4. Définition du tool pour Mistral
# ============================================================

tools = [
    {
        "type": "function",
        "function": {
            "name": "rechercher_evenements",
            "description": (
                "Recherche dans la base des événements culturels "
                "correspondant à la question de l'utilisateur. "
                "Utiliser cette fonction lorsque la question porte "
                "sur des événements, concerts, spectacles, festivals, "
                "expositions, activités, lieux, dates ou horaires."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "question": {
                        "type": "string",
                        "description": (
                            "Question de l'utilisateur à utiliser "
                            "pour effectuer la recherche."
                        ),
                    }
                },
                "required": ["question"],
            },
        },
    }
]


# ============================================================
# 5. Fonction chatbot
# ============================================================

def chatbot(question):
    """
    Envoie une question à Mistral.

    Mistral décide s'il doit utiliser la recherche FAISS.
    Si oui, le résultat de FAISS est renvoyé à Mistral
    afin qu'il formule la réponse finale.
    """

    messages = [
        {
            "role": "system",
            "content": (
                "Tu es un assistant spécialisé dans les événements culturels. "
                "Tu peux rechercher des événements dans une base de données "
                "lorsque cela est nécessaire. "
                "N'invente jamais d'événement, de date, de lieu ou d'horaire. "
                "Lorsque tu utilises la base, construit ta réponse sur les résultats "
                "qui te sont fournis."
            ),
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    # --------------------------------------------------------
    # Premier appel : Mistral analyse la question
    # --------------------------------------------------------

    response = client.chat.complete(
        model="mistral-small-latest",
        messages=messages,
        tools=tools,
        tool_choice="auto",
    )

    message = response.choices[0].message

    # Ajout de la réponse de Mistral à l'historique
    messages.append(message)

    # --------------------------------------------------------
    # Mistral demande-t-il l'utilisation du tool ?
    # --------------------------------------------------------

    if not message.tool_calls:

        # Pas besoin de FAISS :
        return message.content

    # --------------------------------------------------------
    # Exécution des tools demandés par Mistral
    # --------------------------------------------------------

    for tool_call in message.tool_calls:

        if tool_call.function.name == "rechercher_evenements":

            arguments = tool_call.function.arguments

            # Selon la version du SDK, arguments peut être
            # déjà un dictionnaire ou une chaîne JSON.
            if isinstance(arguments, str):
                import json
                arguments = json.loads(arguments)

            resultat = rechercher_evenements(
                arguments["question"]
            )

            # Résultat du tool ajouté à la conversation
            messages.append(
                {
                    "role": "tool",
                    "name": tool_call.function.name,
                    "content": resultat,
                    "tool_call_id": tool_call.id,
                }
            )

    # --------------------------------------------------------
    # Deuxième appel : Mistral formule la réponse finale
    # --------------------------------------------------------

    final_response = client.chat.complete(
        model="mistral-small-latest",
        messages=messages,
    )

    return final_response.choices[0].message.content


In [ ]:
# Tests

question = "Quels concerts sont prévus ce week-end à Vannes ?"

reponse = chatbot(question)

print("QUESTION :")
print(question)

print("\nRÉPONSE :")
print(reponse)

QUESTION :
Quels concerts sont prévus ce week-end ?

RÉPONSE :
Voici les concerts prévus **ce week-end** (11 et 12 octobre 2025) :

---

### **1. LE GRAND SOUFFLET, LES PIEDS DANS L'EAU**
📍 **Lieu** : Prairies St-Martin, quai St-Martin, Rennes (Bretagne)
📅 **Dates** : Samedi 11 octobre 2025
⏰ **Horaires** :
- **10h00 - 20h30** : Concert sur l’eau de *Parveen & Ilyas Khan* (sur un bateau sur le canal).
- **Sur scène** : *Dawndogz* (blues et traditions populaires) et *Maqx* (techno-trad d’Occitanie).

🎵 **Ambiance** : Musique indienne, blues, techno et percussions.
💶 **Tarifs** : Inclut l’aller-retour en bateau pour le concert sur l’eau.

🔗 [Plus d’infos](https://www.rennes-tourisme.com)

---

### **2. Festival Musica Vir'Live**
📍 **Lieu** : Complexe sportif Paul Faubet, Virelade (Nouvelle-Aquitaine)
📅 **Dates** : Vendredi 11 et samedi 12 octobre 2025
⏰ **Horaires** :
- **Vendredi 11/10** : Ouverture à 17h30, concerts dès 18h00 (Be’N Jane, Scène d’été, Batucada Borboleta, Chef and the Ga